# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list the available `RecordSet` entities in the dataset by their `@id`. Each record set's fields and columns (also referenced by `@id`) are shown for comprehensive exploration.

In [ ]:
# List record sets and fields using their '@id'
record_sets = dataset.record_sets
print(f"Total record sets found: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet '@id': {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    field_ids = []
    if 'field' in rs:
        fields = rs['field']
        # fields may be a dict or list 
        if isinstance(fields, dict):
            field_ids = [fields['@id']]
        elif isinstance(fields, list):
            field_ids = [f['@id'] for f in fields]
    print(f"  Fields: {field_ids}")
    col_ids = []
    if 'column' in rs:
        cols = rs['column']
        if isinstance(cols, dict):
            col_ids = [cols['@id']]
        elif isinstance(cols, list):
            col_ids = [c['@id'] for c in cols]
    print(f"  Columns: {col_ids}\n")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.
Use the record set and field `@id`s from the overview.

For demonstration, we extract all record sets and preview the columns and records.

In [ ]:
# Prepare a list of record set '@id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), '\n')

# If there are record sets, select the first one for further demo
if len(record_set_ids) > 0:
    demo_record_set_id = record_set_ids[0]
else:
    demo_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we demonstrate filtering by a numeric column (referenced by its `@id`) from the first record set (if present), normalization, and grouping by a categorical field (also referenced by `@id`).

In [ ]:
# EDA on the first available record set (if any)
if demo_record_set_id:
    df = dataframes[demo_record_set_id]
    print(f"Using RecordSet: {demo_record_set_id}\nColumns: {df.columns.tolist()}\n")

    # Find a numeric column from the record set
    # For demonstration, try 'age' or similar field
    numeric_field_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'duration' in col.lower()]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = 60  # Example threshold for age
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            filtered_df = df[df[numeric_field_id] > threshold]
        else:
            filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        numeric_series = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series - numeric_series.mean()) / numeric_series.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

        # Grouping by categorical field (try 'sex' or 'msi' or similar)
        group_field_candidates = [col for col in df.columns if 'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower()]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouped data by {group_field}:")
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print("No numeric field found in this record set for EDA.")
else:
    print("No record sets found in the dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the numeric field and, if a category is available, a boxplot grouped by category. All fields are referenced via their `@id`.

In [ ]:
import matplotlib.pyplot as plt

if demo_record_set_id and numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]
    numeric_series = pd.to_numeric(df[numeric_field], errors='coerce')

    plt.figure(figsize=(8,4))
    plt.hist(numeric_series.dropna(), bins=15, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field} (by '@id')")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Boxplot, if group field exists
    if group_field_candidates:
        group_field = group_field_candidates[0]
        plt.figure(figsize=(8,4))
        df_plot = df.dropna(subset=[numeric_field, group_field])
        df_plot[numeric_field] = pd.to_numeric(df_plot[numeric_field], errors='coerce')
        df_plot = df_plot[df_plot[numeric_field].notnull()]
        df_plot.boxplot(column=numeric_field, by=group_field)
        plt.title(f"Boxplot of {numeric_field} grouped by {group_field} (by '@id')")
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset contains clinicopathological and MSI-related variables for second primary colorectal cancer in survivors.
- Record sets, fields, and columns are referenced by their `@id` throughout for reproducibility.
- Numeric and categorical variables enable stratified analyses, outlier removal, normalization, and visualization.
- Data supports clinical stratification, biomarker analysis, and informed research for cancer survivors.